In [2]:
# run for only first time

# !git clone https://github.com/thuanz123/realfill.git

In [3]:
# run for only first time
# %cd realfill
# %ls

In [4]:
# !pip install -r /content/realfill/requirements.txt
!pip install -r requirements.txt

In [5]:
from accelerate.utils import write_basic_config
write_basic_config()

/opt/anaconda3/envs/apai3010/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Configuration already exists at /Users/clai404/.cache/huggingface/accelerate/default_config.yaml, will not override. Run `accelerate config` manually or pass a different `save_location`.


False

In [6]:
%pip install "numpy<2"
import numpy
numpy.__version__

Note: you may need to restart the kernel to use updated packages.


'1.26.4'

In [7]:
%pip install diffusers transformers accelerate scipy safetensors 

Note: you may need to restart the kernel to use updated packages.


In [8]:
import torch
from diffusers import StableDiffusionInpaintPipeline

# # Use the 2.0 model you've been working with
# model_id = "sd2-community/stable-diffusion-2-inpainting"

# # Load the pipeline correctly for your Mac
# pipe = StableDiffusionInpaintPipeline.from_pretrained(
#     model_id, 
#     torch_dtype=torch.float32 # Use float32 for Mac stability
# )

local_path = "/Users/clai404/.cache/huggingface/hub/models--sd2-community--stable-diffusion-2-inpainting/snapshots/5f74973cbb64c8568780732c17f43eb269d63a0d"

pipe = StableDiffusionInpaintPipeline.from_pretrained(
    local_path,
    torch_dtype=torch.float32,
    local_files_only=True
)

# Check for Apple Silicon (MPS), NVIDIA (CUDA), or fallback to CPU
if torch.backends.mps.is_available():
    current_device = "mps"
elif torch.cuda.is_available():
    current_device = "cuda"
else:
    current_device = "cpu"

print(f"Applying pipeline to: {current_device}")
pipe = pipe.to(current_device)

# # Use 'mps' for your Apple Silicon Mac
# pipe.to("mps")

# Use your actual variables from the RealFill dataset
# prompt = "Face of a yellow cat, high resolution, sitting on a park bench"
# image = pipe(prompt=prompt, image=image, mask_image=mask_image).images[0]
# image.save("./yellow_cat_on_park_bench.png")

Loading pipeline components...: 100%|██████████| 6/6 [00:00<00:00, 20.01it/s]


Applying pipeline to: mps


In [12]:
# from huggingface_hub import login

# base brain: stable diffusion 2.0 inpainting
%env MODEL_NAME=sd2-community/stable-diffusion-2-inpainting
# reference image
# %env TRAIN_DIR=data/flowerwoman
# %env OUTPUT_DIR=flowerwoman-model

%env TRAIN_DIR=data/chiwah
%env OUTPUT_DIR=chiwah-model

# !accelerate launch train_realfill.py \
!accelerate launch train_realfill.py \
  --pretrained_model_name_or_path=$MODEL_NAME \
  --train_data_dir=$TRAIN_DIR \
  --output_dir=$OUTPUT_DIR \
  --mixed_precision="no" \
  --resolution=512 \
  --train_batch_size=1 \
  --gradient_accumulation_steps=1 \
  --unet_learning_rate=2e-4 \
  --text_encoder_learning_rate=4e-5 \
  --lr_scheduler="constant" \
  --lr_warmup_steps=100 \
  --max_train_steps=2000 \
  --lora_rank=8 \
  --lora_dropout=0.1 \
  --lora_alpha=16

  #   # --train_batch_size=16 \

env: MODEL_NAME=sd2-community/stable-diffusion-2-inpainting
env: TRAIN_DIR=data/chiwah
env: OUTPUT_DIR=chiwah-model
/opt/anaconda3/envs/apai3010/lib/python3.10/site-packages/torchvision/datapoints/__init__.py:12: UserWarning: The torchvision.datapoints and torchvision.transforms.v2 namespaces are still Beta. While we do not expect major breaking changes, some APIs may still change according to user feedback. Please submit any feedback you may have in this issue: https://github.com/pytorch/vision/issues/6753, and you can also check out https://github.com/pytorch/vision/issues/7319 to learn more about the APIs that we suspect might involve future changes. You can silence this warning by calling torchvision.disable_beta_transforms_warning().
  warnings.warn(_BETA_TRANSFORMS_WARNING)
/opt/anaconda3/envs/apai3010/lib/python3.10/site-packages/torchvision/transforms/v2/__init__.py:54: UserWarning: The torchvision.datapoints and torchvision.transforms.v2 namespaces are still Beta. While we do 

In [10]:
%env MODEL_PATH=chiwah-model
%env VAL_IMG=data/chiwah/target/target.png
%env VAL_MASK=data/chiwah/target/mask.png
%env OUTPUT_IMG_DIR=chiwah-results


!accelerate launch infer.py \
    --model_path=$MODEL_PATH \
    --validation_image=$VAL_IMG \
    --validation_mask=$VAL_MASK \
    --output_dir=$OUTPUT_IMG_DIR

env: MODEL_PATH=chiwah-model
env: VAL_IMG=data/chiwah/target/target.png
env: VAL_MASK=data/chiwah/target/mask.png
env: OUTPUT_IMG_DIR=chiwah-results
Using device: mps
Loading pipeline components...: 100%|█████████████| 6/6 [00:00<00:00, 18.98it/s]
Traceback (most recent call last):
  File "/Users/clai404/Desktop/Y3S2/APAI3010 Computer Vision and Image Processing/CVIP GP/realfill/infer.py", line 70, in <module>
    pipe.unet = UNet2DConditionModel.from_pretrained(
  File "/opt/anaconda3/envs/apai3010/lib/python3.10/site-packages/diffusers/models/modeling_utils.py", line 528, in from_pretrained
    config, unused_kwargs, commit_hash = cls.load_config(
  File "/opt/anaconda3/envs/apai3010/lib/python3.10/site-packages/diffusers/configuration_utils.py", line 364, in load_config
    raise EnvironmentError(
OSError: Error no file named config.json found in directory chiwah-model.
Traceback (most recent call last):
  File "/opt/anaconda3/envs/apai3010/bin/accelerate", line 6, in <module>
    s

In [11]:
!zip -r realfill.zip \chiwah-results


  adding: chiwah-results/ (stored 0%)
